# 04 · Train & evaluate PLSR (per sensor location)

Port of `ProBallast_PLSR_02.m`. Builds the labelled table, fits one PLSR per
sensor location (A1, A2, A7, A11), and evaluates on the held-out day-range
test points with MAPE and correlation.

The model is swappable — `fit_per_location` works on a plain (X, y) table, so
XGBoost/LightGBM/CatBoost can drop in later.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import numpy as np, pandas as pd
from intelliswitch import Config
DATA_ROOT = r"/Volumes/Seagate Expansion Drive/Mohammad/M Drive DTU/MATLAB Scripts/IntelliswitchScripts"
W10  = os.path.join(DATA_ROOT, "W10v140")
BLADE = os.path.join(DATA_ROOT, "Bladenr8252")
cfg = Config()


## 1 · Build the labelled training table
Set `N` high (or `None`) to use the full log set. Extraction is ~2 s/log.


In [ ]:
from pathlib import Path
from intelliswitch.features import extract_features
from intelliswitch.geometry import align_campaigns, build_deg_model
from intelliswitch.geometry.deg_model import attach_labels

deg = build_deg_model(align_campaigns(BLADE, cfg), cfg)

N = 300            # number of logs to process (None = all ~7800)
logs = sorted(Path(W10).glob('*.log'))
logs = logs if N is None else logs[:N]
frames = []
for i, p in enumerate(logs):
    r = extract_features(p, cfg)
    if r.accepted:
        frames.append(r.features)
    if (i+1) % 50 == 0:
        print(f'{i+1}/{len(logs)} logs, {sum(len(f) for f in frames)} rows')
data = pd.concat(frames, ignore_index=True)
data = attach_labels(data, deg, cfg)
data = data.dropna(subset=['y'])
print('labelled rows:', len(data))
data.groupby('sensor').size()


## 2 · Optionally persist the table


In [ ]:
# data.to_parquet('training_table.parquet')
data[['source','sensor','days','speed_kmh','y']].describe(include='all')


## 3 · Fit one PLSR per location and evaluate


In [ ]:
from intelliswitch.model import fit_per_location

results = fit_per_location(data, cfg)
summary = []
for name, r in results.items():
    if r['model'] is None:
        summary.append(dict(sensor=name, error=r.get('error'))); continue
    m, e = r['model'], r['eval']
    summary.append(dict(sensor=name, NOC=m.n_components, n_train=m.n_train,
                        train_R2=round(m.train_r2,3),
                        test_pts=e['n_testpoints'],
                        MAPE=round(e['mape'],1), corr=round(e['corr'],3)))
pd.DataFrame(summary)


## 4 · Predicted vs. reference quality at the test points


In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
for ax, (name, r) in zip(axes.ravel(), results.items()):
    if r['model'] is None: ax.set_title(f'{name}: {r.get("error")}'); continue
    e = r['eval']
    ax.plot(e['mid_days'], e['ref'],  '--o', label='reference (geometry)')
    ax.plot(e['mid_days'], e['pred'], '-d',  label='PLSR estimate')
    ax.set_title(f"{name}  MAPE={e['mape']:.1f}%  corr={e['corr']:.2f}")
    ax.set_xlabel('days before 2019-01-01'); ax.grid(True); ax.legend()
plt.tight_layout(); plt.show()
